1. Setup

In [2]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.7 MB/s eta 0:00:00


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
#  removed for securtiy purpose

In [4]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

2. Define  Experts


In [5]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a technical expert. Be precise, code-focused, and help debug issues clearly.",
        "temperature": 0.7
    },
    "billing": {
        "system_prompt": "You are a billing support expert. Be empathetic, explain policies clearly, and help with refunds or charges.",
        "temperature": 0.7
    },
    "general": {
        "system_prompt": "You are a friendly general assistant. Help with casual queries.",
        "temperature": 0.7
    }
}

3. The Router (The Core Task)

In [19]:
def route_prompt(user_input):
    prompt = f"""
Classify this text into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name.

Text: {user_input}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()

In [20]:
def process_request(user_input):
    # Step 1: Route
    category = route_prompt(user_input)

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    config = MODEL_CONFIG[category]

    # Step 2: Call expert
    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=config["temperature"]
    )

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

4. The Orchestrator

In [21]:
def process_request(user_input):
    # Step 1: Route
    category = route_prompt(user_input)

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    config = MODEL_CONFIG[category]

    # Step 2: Call expert
    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": config["system_prompt"]},
        {"role": "user", "content": user_input}
    ],
    temperature=config["temperature"]
)

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

In [22]:
queries = [
    "My Python code gives IndexError",
    "I was charged twice this month",
    "Tell me a joke"
]

for q in queries:
    result = process_request(q)
    print(f"\nQuery: {q}")
    print(f"Category: {result['category']}")
    print(f"Response: {result['response']}")


Query: My Python code gives IndexError
Category: technical
Response: An `IndexError` in Python typically occurs when you try to access an element in a sequence (such as a list, tuple, or string) using an index that is out of range.

To help you debug the issue, I'll need more information about your code. Please provide the following:

1. **The code snippet** that's causing the error
2. **The full error message**, including the line number and the index that's causing the issue
3. **Any relevant context**, such as the data you're working with or the expected output

Once I have this information, I can help you:

* Identify the root cause of the error
* Explain why the error is occurring
* Provide a step-by-step solution to fix the issue
* Offer example code to demonstrate the corrected solution

Please provide the necessary details, and I'll be happy to assist you in debugging your Python code. 

Here is a general example of how to avoid an `IndexError`:
```python
# Example list
my_lis

BONUS

In [23]:
MODEL_CONFIG["tool"] = {
    "system_prompt": "You are a tool router. If query needs real-time data, call tools.",
    "temperature": 0
}

In [28]:
def route_prompt(user_input):
    prompt = f"""
Classify this text into one of these categories:
technical
billing
general
tool

Return ONLY one word.

Text: {user_input}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower().replace(".", "")

In [29]:
def get_bitcoin_price():
    # mock function (no API needed)
    return "Bitcoin price is approximately $65,000 💰 (mock data)"

In [30]:
def process_request(user_input):
    category = route_prompt(user_input)

    # 👉 TOOL HANDLING (comes FIRST)
    if category == "tool":
        if "bitcoin" in user_input.lower():
            return {
                "category": "tool",
                "response": get_bitcoin_price()
            }
        else:
            return {
                "category": "tool",
                "response": "Tool not available for this query."
            }

    # 👉 NORMAL FLOW
    if category not in MODEL_CONFIG:
        category = "general"

    config = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=config["temperature"]
    )

    return {
        "category": category,
        "response": response.choices[0].message.content
    }

In [31]:
queries = [
    "What is the current price of Bitcoin?",
    "Tell me a joke",
    "My code has an error"
]

for q in queries:
    result = process_request(q)
    print(f"\nQuery: {q}")
    print(f"Category: {result['category']}")
    print(f"Response: {result['response']}")


Query: What is the current price of Bitcoin?
Category: general
Response: I'm a large language model, I don't have real-time access to current market prices. However, I can suggest some ways for you to find the current price of Bitcoin:

1. Check online cryptocurrency exchanges: Websites like Coinbase, Binance, or Kraken usually display the current price of Bitcoin.
2. Use a cryptocurrency price tracker: Websites like CoinMarketCap or CoinGecko provide real-time price updates for various cryptocurrencies, including Bitcoin.
3. Download a cryptocurrency app: Many cryptocurrency apps, such as Blockfolio or CryptoPro, provide real-time price updates and can be downloaded on your mobile device.

Please note that the price of Bitcoin can fluctuate rapidly, so it's essential to check a reliable source for the most up-to-date information.

Query: Tell me a joke
Category: general
Response: Here's one: What do you call a fake noodle?

An impasta.

Query: My code has an error
Category: technical